# SolanaSentinel — Hyperparameter Tuning con Optuna (v2)

Optimizamos **PR-AUC** (metrica correcta para datasets desbalanceados) para tres modelos.
Usa el mismo label que 02_train v2: `outcome_max_gain_pct >= 20` (TP-aligned).

- Validacion con **TimeSeriesSplit** para no filtrar datos futuros al pasado.
- El modelo tuneado se guarda como `pump_predictor_<nombre>_tuned_v2.joblib`.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve
)

import xgboost as xgb
import lightgbm as lgb

plt.style.use('dark_background')
ACCENT = '#00d4ff'
GREEN  = '#00ff88'
RED    = '#ff4444'
PURPLE = '#a78bfa'
YELLOW = '#fbbf24'
ORANGE = '#fb923c'

DB_PATH    = Path('../backend/data/sentinel.db')
MODELS_DIR = Path('../backend/data/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
Path('charts').mkdir(exist_ok=True)

LABEL_THRESHOLD = 20.0
LABEL_COL       = 'outcome_max_gain_pct'
BC_GRADUATION   = 85.0
N_TRIALS        = 60    # trials por modelo (aumentar para mejor resultado)
N_CV_SPLITS     = 4     # folds temporales para cross-validation

assert DB_PATH.exists(), f'DB no encontrada: {DB_PATH}'
print(f'Optuna {optuna.__version__} | XGBoost {xgb.__version__} | LightGBM {lgb.__version__}')
print(f'Trials por modelo: {N_TRIALS} | CV splits: {N_CV_SPLITS}')

## 1. Carga y features (igual que 02_train)

In [ ]:
conn = sqlite3.connect(DB_PATH)
df_raw = pd.read_sql_query("""
    SELECT id, source, detected_at,
           initial_liquidity, market_cap, volume_1h,
           COALESCE(NULLIF(price_usd, 0), latest_price_usd) AS price_usd,
           bonding_curve_complete, bonding_curve_real_sol,
           bonding_curve_mc_sol, bonding_curve_price_sol,
           risk_score, risk_level,
           outcome_max_gain_pct, outcome_price_1h
    FROM detected_tokens
    WHERE outcome_complete = 1
      AND outcome_max_gain_pct IS NOT NULL
""", conn)
conn.close()

eps = 1e-12
df  = df_raw.copy()

df['label']    = (df['outcome_max_gain_pct'] >= LABEL_THRESHOLD).astype(int)

df['f_log_mc']        = np.log1p(df['market_cap'].fillna(0).clip(lower=0))
df['f_log_liq']       = np.log1p(df['initial_liquidity'].fillna(0).clip(lower=0))
df['f_log_price']     = np.log1p(df['price_usd'].fillna(0).clip(lower=0))
df['f_liq_mc_ratio']  = df['initial_liquidity'].fillna(0) / (df['market_cap'].fillna(0) + eps)
df['f_on_curve']      = (df['bonding_curve_complete'].fillna(1) == 0).astype(float)
df['f_log_bc_sol']    = np.log1p(df['bonding_curve_real_sol'].fillna(0).clip(lower=0))
df['f_log_bc_mc_sol'] = np.log1p(df['bonding_curve_mc_sol'].fillna(0).clip(lower=0))
df['f_risk_score']    = df['risk_score'].fillna(50)
df['f_is_safe']       = (df['risk_level'].isin(['safe', 'low'])).astype(float)
df['f_is_pumpfun']    = (df['source'] == 'pumpfun').astype(float)
df['detected_dt']     = pd.to_datetime(df['detected_at'])
df['f_hour_sin']      = np.sin(2 * np.pi * df['detected_dt'].dt.hour / 24)
df['f_hour_cos']      = np.cos(2 * np.pi * df['detected_dt'].dt.hour / 24)
df['f_bc_progress']   = (df['bonding_curve_real_sol'].fillna(0) / BC_GRADUATION).clip(0, 1)
df['f_dow_sin']       = np.sin(2 * np.pi * df['detected_dt'].dt.dayofweek / 7)
df['f_dow_cos']       = np.cos(2 * np.pi * df['detected_dt'].dt.dayofweek / 7)

FEATURE_COLS = [
    'f_log_mc', 'f_log_liq', 'f_log_price', 'f_liq_mc_ratio',
    'f_on_curve', 'f_log_bc_sol', 'f_log_bc_mc_sol',
    'f_risk_score', 'f_is_safe', 'f_is_pumpfun',
    'f_hour_sin', 'f_hour_cos',
    'f_bc_progress', 'f_dow_sin', 'f_dow_cos',
]

df_s    = df.sort_values('detected_at').reset_index(drop=True)
X       = df_s[FEATURE_COLS].fillna(0).values
y       = df_s['label'].values
split   = int(len(df_s) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

n_pos    = y_train.sum()
n_neg    = (y_train == 0).sum()
base     = y_test.mean()
spw      = n_neg / max(n_pos, 1)

print(f'Total:      {len(df_s):,}')
print(f'Train:      {len(X_train):,}  ->  {n_pos} pos ({y_train.mean()*100:.1f}%)')
print(f'Test:       {len(X_test):,}   ->  {y_test.sum()} pos ({base*100:.1f}%)')
print(f'Imbalance:  {spw:.1f}:1  ->  scale_pos_weight = {spw:.1f}')
print(f'Base rate (test): {base*100:.2f}%  |  objetivo PR-AUC > {base:.3f}')

## 2. Funciones de utilidad Optuna

In [ ]:
from sklearn.metrics import average_precision_score

tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)

def cv_pr_auc(model, X_tr, y_tr):
    """PR-AUC medio en cross-validation temporal."""
    scores = []
    for tr_idx, val_idx in tscv.split(X_tr):
        model.fit(X_tr[tr_idx], y_tr[tr_idx])
        proba = model.predict_proba(X_tr[val_idx])[:, 1]
        scores.append(average_precision_score(y_tr[val_idx], proba))
    return float(np.mean(scores))

def eval_final(model, X_tr, y_tr, X_te, y_te, name):
    """Entrena en todo el train y evalúa en test."""
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    auc   = roc_auc_score(y_te, proba)
    ap    = average_precision_score(y_te, proba)
    lift  = ap / base if base > 0 else 0
    print(f'{name:<25}  ROC-AUC={auc:.4f}  PR-AUC={ap:.4f}  lift={lift:.1f}x')
    return proba, auc, ap

print('Utilidades listas.')

## 3. Tuning — Random Forest

In [ ]:
def rf_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 200, 800, step=100),
        max_depth         = trial.suggest_int('max_depth', 4, 16),
        min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 20),
        max_features      = trial.suggest_float('max_features', 0.3, 1.0),
        class_weight      = 'balanced_subsample',
        random_state      = 42,
        n_jobs            = -1,
    )
    return cv_pr_auc(RandomForestClassifier(**params), X_train, y_train)

print(f'Optimizando Random Forest ({N_TRIALS} trials)...')
study_rf = optuna.create_study(direction='maximize',
                               sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(rf_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_rf_params = study_rf.best_params
best_rf_params.update({'class_weight': 'balanced_subsample', 'random_state': 42, 'n_jobs': -1})
print(f'\nBest CV PR-AUC: {study_rf.best_value:.4f}')
print(f'Best params: {best_rf_params}')

## 4. Tuning — XGBoost

In [ ]:
def xgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 200, 1000, step=100),
        max_depth         = trial.suggest_int('max_depth', 3, 10),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_weight  = trial.suggest_int('min_child_weight', 1, 20),
        gamma             = trial.suggest_float('gamma', 0, 5),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        scale_pos_weight  = spw,
        eval_metric       = 'aucpr',
        random_state      = 42,
        n_jobs            = -1,
        verbosity         = 0,
    )
    return cv_pr_auc(xgb.XGBClassifier(**params), X_train, y_train)

print(f'Optimizando XGBoost ({N_TRIALS} trials)...')
study_xgb = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
best_xgb_params.update({'scale_pos_weight': spw, 'eval_metric': 'aucpr',
                         'random_state': 42, 'n_jobs': -1, 'verbosity': 0})
print(f'\nBest CV PR-AUC: {study_xgb.best_value:.4f}')
print(f'Best params: {best_xgb_params}')

## 5. Tuning — LightGBM

In [ ]:
def lgb_objective(trial):
    params = dict(
        n_estimators      = trial.suggest_int('n_estimators', 200, 1000, step=100),
        max_depth         = trial.suggest_int('max_depth', 3, 12),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 15, 127),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 10, log=True),
        scale_pos_weight  = spw,
        random_state      = 42,
        n_jobs            = -1,
        verbose           = -1,
    )
    return cv_pr_auc(lgb.LGBMClassifier(**params), X_train, y_train)

print(f'Optimizando LightGBM ({N_TRIALS} trials)...')
study_lgb = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(lgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

best_lgb_params = study_lgb.best_params
best_lgb_params.update({'scale_pos_weight': spw, 'random_state': 42, 'n_jobs': -1, 'verbose': -1})
print(f'\nBest CV PR-AUC: {study_lgb.best_value:.4f}')
print(f'Best params: {best_lgb_params}')

## 6. Evaluación final en test + comparación

In [ ]:
tuned_models = {
    'RF_tuned':  RandomForestClassifier(**best_rf_params),
    'XGB_tuned': xgb.XGBClassifier(**best_xgb_params),
    'LGB_tuned': lgb.LGBMClassifier(**best_lgb_params),
}

results = {}
print('=== Evaluación final en test set ===')
print(f'Base rate: {base*100:.2f}%')
print()
for name, model in tuned_models.items():
    proba, auc, ap = eval_final(model, X_train, y_train, X_test, y_test, name)
    results[name] = {'model': model, 'proba': proba, 'auc': auc, 'ap': ap}

best_name  = max(results, key=lambda k: results[k]['ap'])
best_model = results[best_name]['model']
best_proba = results[best_name]['proba']
print(f'\nMejor modelo (PR-AUC): {best_name} = {results[best_name]["ap"]:.4f}')

# Compare against 02_train baseline (run just before this cell)
baseline_02 = max(r['ap'] for r in results.values())  # updated dynamically
print(f'\nRanking de modelos tuneados:')
for name, r in sorted(results.items(), key=lambda x: -x[1]['ap']):
    star = ' ← MEJOR' if name == best_name else ''
    print(f'  {name:<20}  ROC-AUC={r["auc"]:.4f}  PR-AUC={r["ap"]:.4f}  lift={r["ap"]/base:.1f}x{star}')

In [ ]:
colors = [ACCENT, GREEN, PURPLE, YELLOW]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- ROC ---
ax = axes[0]
ax.plot([0,1],[0,1],'w--',lw=0.8,label='Random (0.50)')
for (name, r), c in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['proba'])
    ax.plot(fpr, tpr, color=c, lw=2, label=f'{name} ({r["auc"]:.3f})')
ax.set_title('ROC Curve'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# --- PR ---
ax = axes[1]
ax.axhline(base, color='white', ls='--', lw=0.8, label=f'Random ({base:.3f})')
for (name, r), c in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, r['proba'])
    ax.plot(rec, prec, color=c, lw=2, label=f'{name} ({r["ap"]:.3f})')
ax.set_title('Precision-Recall Curve'); ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# --- Optuna convergence ---
ax = axes[2]
for study, name, c in [(study_rf,'RF',ACCENT),(study_xgb,'XGB',GREEN),(study_lgb,'LGB',PURPLE)]:
    vals  = [t.value for t in study.trials if t.value is not None]
    bests = np.maximum.accumulate(vals)
    ax.plot(range(1, len(bests)+1), bests, color=c, lw=2, label=name)
ax.set_title('Optuna — PR-AUC convergencia')
ax.set_xlabel('Trial'); ax.set_ylabel('Best PR-AUC (CV)')
ax.legend(); ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('charts/20_tuning_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Win rate por decil — mejor modelo tuneado

In [ ]:
df_eval = pd.DataFrame({'score': best_proba, 'pumped': y_test,
                         'ratio': df_s.iloc[split:]['ratio_1h'].values})
df_eval['decile'] = pd.qcut(df_eval['score'], q=10, labels=False, duplicates='drop')

dec = df_eval.groupby('decile').agg(
    n=('pumped','count'), win_rate=('pumped','mean'),
    score_min=('score','min'), score_max=('score','max')
).reset_index()
dec['range'] = dec.apply(lambda r: f'{r.score_min*100:.0f}-{r.score_max*100:.0f}', axis=1)

target_3x = base * 3
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
bar_colors = [GREEN if w >= target_3x else ACCENT if w >= base else RED
              for w in dec['win_rate']]
bars = ax.bar(range(len(dec)), dec['win_rate']*100, color=bar_colors, edgecolor='none', alpha=0.85)
ax.axhline(base*100, color='white', ls='--', lw=1.5, label=f'Base rate ({base*100:.1f}%)')
ax.axhline(target_3x*100, color=GREEN, ls=':', lw=1, label=f'3x base ({target_3x*100:.1f}%)')
ax.set_xticks(range(len(dec)))
ax.set_xticklabels(dec['range'], rotation=45, ha='right', fontsize=8)
ax.set_title(f'Win rate por decil — {best_name} (tuneado)')
ax.set_ylabel('% tokens que suben +20% en 1h')
for bar, val in zip(bars, dec['win_rate']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val*100:.0f}%', ha='center', va='bottom', fontsize=8)
ax.legend()

# Optuna param importance para el mejor modelo
ax = axes[1]
study_map = {'RF_tuned': study_rf, 'XGB_tuned': study_xgb, 'LGB_tuned': study_lgb}
best_study = study_map[best_name]
try:
    imp = optuna.importance.get_param_importances(best_study)
    params_names = list(imp.keys())[:8]
    params_vals  = [imp[k] for k in params_names]
    ax.barh(params_names[::-1], params_vals[::-1], color=ACCENT, edgecolor='none')
    ax.set_title(f'Importancia de hiperparámetros — {best_name}')
    ax.set_xlabel('Importancia relativa')
except Exception as e:
    ax.text(0.5, 0.5, f'No disponible: {e}', transform=ax.transAxes, ha='center')

plt.tight_layout()
plt.savefig('charts/21_tuned_winrate.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nWin rate por decil:')
for _, row in dec.iterrows():
    flag = ' ✅' if row.win_rate >= target_3x else ''
    print(f'  {row["range"]:<12}  n={int(row.n):<5}  win={row.win_rate*100:.1f}%{flag}')

## 8. Threshold y guardado del mejor modelo

In [ ]:
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, best_proba)
target_prec = base * 3
valid       = prec_arr[:-1] >= target_prec
threshold   = float(thresh_arr[valid][0]) if valid.any() else 0.5
signals     = (best_proba >= threshold).sum()

print(f'Threshold (3x base = {target_prec*100:.1f}% precision): {threshold:.4f}')
print(f'Señales en test: {signals} / {len(y_test)} ({signals/len(y_test)*100:.1f}%)')

# Guardar
model_file = f'pump_predictor_{best_name.lower()}.joblib'
joblib.dump(best_model, MODELS_DIR / model_file)

meta = {
    'model_name':      best_name,
    'model_file':      model_file,
    'feature_cols':    FEATURE_COLS,
    'threshold':       threshold,
    'label':           'pump_20pct_1h',
    'label_threshold': float(LABEL_THRESHOLD),
    'roc_auc':         float(results[best_name]['auc']),
    'pr_auc':          float(results[best_name]['ap']),
    'base_rate':       float(base),
    'train_samples':   int(len(X_train)),
    'test_samples':    int(len(X_test)),
    'tuned':           True,
    'tuner':           'optuna',
    'n_trials':        N_TRIALS,
    'best_params':     study_map[best_name].best_params,
}
with open(MODELS_DIR / 'model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print()
print('=' * 56)
print('  RESUMEN TUNING')
print('=' * 56)
for name, r in sorted(results.items(), key=lambda x: -x[1]['ap']):
    delta = r['ap'] - 0.245
    sign  = '+' if delta >= 0 else ''
    star  = ' ← MEJOR' if name == best_name else ''
    print(f'  {name:<20}  ROC-AUC={r["auc"]:.3f}  PR-AUC={r["ap"]:.3f}  ({sign}{delta:.3f} vs baseline){star}')
print(f'\n  Guardado: {model_file}')
print('=' * 56)
print()
print('Proximo paso: 04_advanced_models.ipynb (TabNet + MLP)')